# Advanced RAG: Zettelkasten & GraphRAG

Welcome to Part 2! In this notebook, we will upgrade our RAG pipeline from a basic vector search to a **state-aware Agentic Memory system**, inspired by the Zettelkasten "slip-box" method.

### Learning Objectives:
1. **Contextual Retrieval**: Use Gemini to augment chunks with overarching document context.
2. **GraphRAG Entity Extraction**: Use Gemini Structured Outputs to extract Entities and Relationships, forming a Knowledge Graph.
3. **Topological Traversal**: Traverse the graph during retrieval to find logically connected concepts that vector search misses.
4. **Stateful Compilation**: Compile the raw chunks and extracted connections into physical Markdown files (an LLM Wiki).

## Setup and Dependencies
We'll need `pydantic` for structured outputs and `networkx` for our local Knowledge Graph.

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import networkx as nx

load_dotenv()
client = genai.Client()
print("Gemini API Key successfully loaded and Client initialized.")

Gemini API Key successfully loaded and Client initialized.


## Step 1: Contextual Chunking
A key flaw of recursive chunking is that it isolates text from its broader narrative (e.g., a chunk saying \"It cost $2M\" is useless if the previous chunk named the project).
We solve this via **Contextual Retrieval**: an LLM reads the full document and generates a brief contextual summary for the specific chunk before indexing it.

In [2]:
def split_text_basic(text, chunk_size=1200, chunk_overlap=300):
    """Basic chunker (re-used from Part 1)"""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end].strip())
        start = end - chunk_overlap
        if start >= len(text) or end == len(text):
            break
    return chunks

def augment_chunk_with_context(client, document_text, chunk_text):
    """Uses Gemini to prepend context to a chunk."""
    prompt = (
        f"You are an expert document archivist.\n"
        f"Below is a full document, followed by a small chunk extracted from it.\n"
        f"Your task is to write a succinct (1-2 sentences) context statement that explains how the chunk fits into the broader document.\n"
        f"---\nFull Document:\n{document_text[:4000]}... (truncated)\n"
        f"---\nChunk:\n{chunk_text}\n"
        f"---\nContext Statement:"
    )
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return f"[Context: {response.text.strip()}]\n{chunk_text}"

# Note: Running this on thousands of chunks is expensive! For this tutorial, we will only process a single document.
print("Contextual Chunking functions defined.")

Contextual Chunking functions defined.


## Step 2: GraphRAG Entity Extraction (Building the Zettelkasten)
Instead of just throwing chunks into a vector database, we want to extract the explicit *concepts* (Nodes) and how they relate (Edges). We use Gemini's **Structured Outputs**.

In [3]:
class Node(BaseModel):
    name: str = Field(description="The name of the entity, concept, or tool (e.g., 'vLLM', 'RAG', 'Andrej Karpathy')")
    type: str = Field(description="Type of entity: Tool, Concept, Person, Organization, etc.")
    description: str = Field(description="Brief definition or context of this entity in the text.")

class Edge(BaseModel):
    source: str = Field(description="Name of the source node")
    target: str = Field(description="Name of the target node")
    relationship: str = Field(description="How they relate (e.g., 'DEPENDS_ON', 'CREATED_BY', 'CONTRADICTS')")

class KnowledgeGraph(BaseModel):
    nodes: list[Node]
    edges: list[Edge]

def extract_graph(client, chunk_text):
    """Extracts nodes and edges from a text chunk using Gemini."""
    prompt = (
        f"Extract a knowledge graph from the following text.\n"
        f"Identify key technical concepts, tools, and organizations as nodes.\n"
        f"Identify the logical relationships between them as edges.\n"
        f"Text:\n{chunk_text}"
    )
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=KnowledgeGraph,
            temperature=0.0
        )
    )
    return KnowledgeGraph.model_validate_json(response.text)

print("Graph extraction schema and function defined.")

Graph extraction schema and function defined.


## Step 3: Process a Document and Build the Local Graph
Let's load a single source file, contextually augment its chunks, extract the graph elements, and build a `networkx` graph.

In [4]:
# Load a single document for testing
source_file = list(Path("data/sources").glob("*.txt"))[0]
with open(source_file, "r") as f:
    full_text = f.read()

print(f"Processing: {source_file.name}")

# Take just the first 2 chunks to keep API costs/time low for the tutorial
raw_chunks = split_text_basic(full_text)[:2]
augmented_chunks = []
extracted_graphs = []

for i, chunk in enumerate(raw_chunks):
    print(f"\n--- Processing Chunk {i+1} ---")
    # 1. Contextual Augmentation
    aug_chunk = augment_chunk_with_context(client, full_text, chunk)
    augmented_chunks.append(aug_chunk)
    print("Augmented Chunk:\n", aug_chunk[:150], "...")
    
    # 2. Graph Extraction
    graph_data = extract_graph(client, aug_chunk)
    extracted_graphs.append(graph_data)
    print(f"Extracted {len(graph_data.nodes)} nodes and {len(graph_data.edges)} edges.")

# Build the NetworkX Graph
G = nx.Graph()
for g in extracted_graphs:
    for node in g.nodes:
        # Lowercase for simple deduplication
        G.add_node(node.name.lower(), type=node.type, description=node.description)
    for edge in g.edges:
        G.add_edge(edge.source.lower(), edge.target.lower(), relationship=edge.relationship)

print(f"\nLocal Knowledge Graph built with {G.number_of_nodes()} total unique nodes and {G.number_of_edges()} edges.")

Processing: LLMOps Course _ Deploy _ Scale Production LLMs.txt

--- Processing Chunk 1 ---
Augmented Chunk:
 [Context: This initial chunk serves as the document's introductory section, presenting the LLMOps course title, providing navigation links for the Sch ...
Extracted 26 nodes and 31 edges.

--- Processing Chunk 2 ---
Augmented Chunk:
 [Context: This chunk defines the LLMOps course, detailing its structure, practical components, and key features like duration and cost, while also inc ...
Extracted 33 nodes and 32 edges.

Local Knowledge Graph built with 45 total unique nodes and 51 edges.


## Step 4: Topological Traversal (Graph Hop)
Now, imagine a user asks about a specific concept. Traditional RAG finds chunks that *mention* the concept.
GraphRAG finds the concept in the graph, and traverses the *edges* to pull in logically related concepts, even if they aren't mentioned in the same paragraph.

In [5]:
def traverse_graph(graph, start_node_name, depth=1):
    """Finds a node and returns its neighbors up to N hops away."""
    start_node = start_node_name.lower()
    if start_node not in graph.nodes:
        # In a real system, you'd use vector search to find the closest node (Anchor Search)
        return f"Node '{start_node}' not found in the graph."
    
    # Get subgraph of neighbors within 'depth'
    neighbors = nx.single_source_shortest_path_length(graph, start_node, cutoff=depth)
    subgraph = graph.subgraph(neighbors.keys())
    
    result = f"Topological context for '{start_node}':\n"
    for u, v, data in subgraph.edges(data=True):
        result += f" - {u} [{data.get('relationship', 'RELATES_TO')}] {v}\n"
    return result

# Let's view the nodes we have to pick one
print("Available nodes in our mini-graph:")
print(list(G.nodes)[:10])

# Example traversal (Pick a node name that printed above!)
if G.number_of_nodes() > 0:
    example_node = list(G.nodes)[0]
    traversal_result = traverse_graph(G, example_node)
    print("\n=== Graph Traversal Results ===")
    print(traversal_result)

Available nodes in our mini-graph:
['llmops course', 'llmops (large language model operations)', 'school of core ai', 'production llms', 'serving', 'observability', 'evaluation gates', 'secure releases', 'cost control', 'vllm']

=== Graph Traversal Results ===
Topological context for 'llmops course':
 - secure releases [COVERS_TOPIC] llmops course
 - secure releases [ENCOMPASSES] llmops (large language model operations)
 - langserve [UTILIZES] llmops course
 - school of core ai [PROVIDED_BY] llmops course
 - mlflow [UTILIZES] llmops course
 - langsmith [UTILIZES] llmops course
 - observability [COVERS_TOPIC] llmops course
 - observability [ENCOMPASSES] llmops (large language model operations)
 - cost control [COVERS_TOPIC] llmops course
 - cost control [ENCOMPASSES] llmops (large language model operations)
 - evaluation gates [COVERS_TOPIC] llmops course
 - evaluation gates [INCLUDES] infra artifacts
 - evaluation gates [ENCOMPASSES] llmops (large language model operations)
 - evaluati

## Step 5: Stateful LLM Wiki Compilation
A true Zettelkasten is persistent. Instead of just returning a chat message, we write these entities out as Markdown files with `[[wikilinks]]` so they compound over time.

In [6]:
wiki_dir = Path("data/wiki")
wiki_dir.mkdir(parents=True, exist_ok=True)

print("Compiling Graph into LLM Wiki...")
for node_id in G.nodes:
    node_data = G.nodes[node_id]
    # Find all edges connected to this node to create wikilinks
    edges = list(G.edges(node_id, data=True))
    
    markdown_content = f"# {node_id.title()}\n\n"
    markdown_content += f"**Type**: {node_data.get('type', 'Unknown')}\n\n"
    markdown_content += f"## Description\n{node_data.get('description', '')}\n\n"
    markdown_content += f"## Logical Connections\n"
    
    for u, v, data in edges:
        # If the edge is connected to us, link the other node
        other_node = v if u == node_id else u
        rel = data.get('relationship', 'RELATES_TO')
        markdown_content += f"- {rel}: [[{other_node.title()}]]\n"
    
    # Sanitize filename
    safe_filename = "".join([c for c in node_id if c.isalpha() or c.isdigit() or c==' ']).rstrip().replace(' ', '_')
    if not safe_filename:
        continue
        
    file_path = wiki_dir / f"{safe_filename}.md"
    with open(file_path, "w") as f:
        f.write(markdown_content)

print(f"Wiki compilation complete! Check the '{wiki_dir}' folder.")
for f in list(wiki_dir.glob("*.md"))[:5]:
    print(f" - {f.name}")

Compiling Graph into LLM Wiki...
Wiki compilation complete! Check the 'data/wiki' folder.
 - kubernetes.md
 - llmops.md
 - concurrency_limits.md
 - engineers.md
 - guardrails.md
